# SSTW motion threshold calibration Colab 入口

该 Notebook 只负责冻结 motion threshold。它会在冷启动 Colab 中挂载 Google Drive、拉取代码、运行 Wan2.1 calibration split、计算 formal motion metrics, 并写出阈值 artifact。


In [ ]:
# 1. 挂载 Google Drive 并检查 GPU
from google.colab import drive
drive.mount('/content/drive')

import json
import os
import subprocess
import sys
from pathlib import Path

print(subprocess.getoutput('nvidia-smi'))
DRIVE_PROJECT_ROOT = os.environ.get('SSTW_DRIVE_PROJECT_ROOT', '/content/drive/MyDrive/SSTW')
Path(DRIVE_PROJECT_ROOT).mkdir(parents=True, exist_ok=True)
print('drive_project_root =', DRIVE_PROJECT_ROOT)


In [ ]:
# 2. 冷启动获取仓库代码
# Colab 环境不会保留代码, 每次运行都需要 clone 或 pull。
REPO_URL = os.environ.get('SSTW_REPO_URL', 'https://github.com/RICHAAARC/SSTW.git')
REPO_DIR = os.environ.get('SSTW_REPO_DIR', '/content/SSTW')

if not Path(REPO_DIR).exists():
    if not REPO_URL:
        raise RuntimeError('请先设置 SSTW_REPO_URL, 或将仓库上传到 /content/SSTW')
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    print('仓库目录已存在, 执行 git pull 以同步远端代码。')
    %cd "$REPO_DIR"
    !git pull --ff-only
%cd "$REPO_DIR"
!git rev-parse --short HEAD


In [ ]:
# 3. 安装 Colab GPU 运行依赖
%pip install -U git+https://github.com/huggingface/diffusers transformers accelerate safetensors imageio imageio-ffmpeg sentencepiece protobuf huggingface_hub


In [ ]:
# 4. 可选 Hugging Face 认证
# token 只写入当前 Colab 环境变量, 不写入 records、tables、reports 或 package manifest。
try:
    from huggingface_hub import login
except Exception as exc:
    raise RuntimeError('huggingface_hub 未安装或无法导入, 请重新执行依赖安装单元') from exc

if os.environ.get('HF_TOKEN'):
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    print('HF_TOKEN 状态: provided from environment.')
else:
    print('HF_TOKEN 状态: not_provided in environment, 将尝试匿名下载公开模型。')


In [ ]:
# 5. 读取统一 workflow profile 配置并初始化 Drive 布局
from paper_workflow.notebook_utils import generative_video_model_probe_workflow as probe_workflow

NOTEBOOK_ROLE = 'motion_threshold_calibration'
WORKFLOW_PROFILE = os.environ.get('SSTW_WORKFLOW_PROFILE') or probe_workflow.default_workflow_profile_for_notebook_role(NOTEBOOK_ROLE)
workflow_profile = probe_workflow.resolve_notebook_workflow_profile(WORKFLOW_PROFILE, NOTEBOOK_ROLE)
PROFILE = workflow_profile['runtime_profile']
layout = probe_workflow.ensure_drive_layout(
    DRIVE_PROJECT_ROOT,
    workflow_profile=WORKFLOW_PROFILE,
    notebook_role=NOTEBOOK_ROLE,
)

print(json.dumps({
    'workflow_profile': workflow_profile,
    'layout': layout,
    'enabled_stage_plan': probe_workflow.build_workflow_stage_plan(WORKFLOW_PROFILE, NOTEBOOK_ROLE),
}, ensure_ascii=False, indent=2))


In [ ]:
# 6. 通用执行 helper
# Notebook 只调用仓库模块, 不在 cell 中手写正式 records、tables、figures 或 reports。
def stage_enabled(stage_name: str) -> bool:
    return probe_workflow.workflow_stage_enabled(WORKFLOW_PROFILE, NOTEBOOK_ROLE, stage_name)


def run_or_raise(stage_name: str, command: list[str]) -> None:
    print('\n===== stage:', stage_name, '=====')
    print(' '.join(command))
    result = probe_workflow.run_command(command)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise SystemExit(result.returncode)


In [ ]:
# 7. 配置模型与质量评估参数
# 这些参数只定义运行入口; 正式协议、样本规模和输出路径由 workflow profile 配置控制。
MODEL_ID = os.environ.get('SSTW_MODEL_ID', 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers')
CROSS_MODEL_ID = os.environ.get('SSTW_CROSS_MODEL_ID', '')
SEMANTIC_MODEL_ID = os.environ.get('SSTW_SEMANTIC_MODEL_ID', 'openai/clip-vit-base-patch32')
SEMANTIC_FRAME_LIMIT = int(os.environ.get('SSTW_SEMANTIC_FRAME_LIMIT', '8'))
DISABLE_SEMANTIC_METRIC = os.environ.get('SSTW_DISABLE_SEMANTIC_METRIC', 'false').lower() == 'true'
INCLUDE_VIDEOS_IN_PACKAGE = os.environ.get('SSTW_INCLUDE_VIDEOS_IN_PACKAGE', 'true').lower() == 'true'


In [ ]:
# 8. 生成 prompt suite、运行 Wan2.1、计算 formal metrics
if stage_enabled('prepare_prompt_suite'):
    run_or_raise('prepare_prompt_suite', probe_workflow.build_prompt_suite_command(layout))

if stage_enabled('wan21_runtime_generation'):
    run_or_raise('wan21_runtime_generation', probe_workflow.build_colab_runtime_command(layout, PROFILE, MODEL_ID, CROSS_MODEL_ID))

if stage_enabled('formal_metric_scoring'):
    run_or_raise(
        'formal_metric_scoring',
        probe_workflow.build_formal_metric_command(
            layout,
            semantic_model_id=SEMANTIC_MODEL_ID,
            semantic_frame_limit=SEMANTIC_FRAME_LIMIT,
            disable_semantic_metric=DISABLE_SEMANTIC_METRIC,
        ),
    )


In [ ]:
# 9. 执行或复用 motion threshold calibration
# 非 calibration profile 只能复用已通过的 calibration artifact, 不能用 evaluation 样本重新估计阈值。
if stage_enabled('motion_threshold_calibration'):
    run_or_raise('motion_threshold_calibration', probe_workflow.build_motion_threshold_calibration_command(layout))

if stage_enabled('motion_threshold_reuse_check') or stage_enabled('motion_threshold_calibration_or_reuse_check'):
    if workflow_profile['workflow_profile'] == 'motion_calibration':
        if stage_enabled('motion_threshold_calibration_or_reuse_check'):
            run_or_raise('motion_threshold_calibration', probe_workflow.build_motion_threshold_calibration_command(layout))
    else:
        motion_threshold_reuse = probe_workflow.validate_motion_threshold_ready_for_profile(layout, WORKFLOW_PROFILE)
        print(json.dumps(motion_threshold_reuse, ensure_ascii=False, indent=2))


In [ ]:
# 末尾治理审计与 Google Drive 打包
if stage_enabled('quick_tests_and_harness'):
    !pytest -q
    !python tools/harness/run_all_audits.py

if stage_enabled('drive_packaging'):
    run_or_raise('drive_packaging', probe_workflow.build_drive_packaging_command(layout, include_videos=INCLUDE_VIDEOS_IN_PACKAGE))

package_dir = Path(layout['drive_package_dir'])
print('package_dir =', package_dir)
if package_dir.exists():
    for path in sorted(package_dir.glob('*')):
        print(path)
